# DeepMesh 시나리오 데모

| 시나리오 | 재현 내용 |
|---|---|
| benign | 유효 토큰 검증, 글 존재 확인, 익명 refresh, SPA 응답 |
| l2 | 서명이 틀리거나 형식이 잘못된 토큰의 인증 시도 |
| enum_seq | 테스트 글의 댓글 순차 조회. 정상 탐색과도 겹침 |
| l3 | 테스트 댓글의 실제 일괄 삭제 |
| cred_enum | 소수 계정에 대한 비밀번호 추측과 실패 메시지 비교 |
| scan_seq / mixed | 민감 경로 탐색과 정상 조회의 혼합 |
| r1 | 정적 HTML 변조와 스크립트 삽입 응답 |
| k1 / k2 | Kubernetes API 조회 / 리소스 조작 dry-run |
| SSH 모의 | 지정한 실습 서버에 연결해 SSH 배너 전송 |

테스트 데이터 생성·정리도 트래픽을 만든다. 시나리오 출력의 시작·종료 시각으로 대시보드 구간을 구분해야 한다.

## 1. 접속 설정

테스트 계정의 실제 access token을 입력한다. 로그인 응답 JSON 전체를 붙여 넣어도 된다.
JWT 형식이 아니거나 서버가 401을 반환하면 새 토큰을 다시 묻는다. 토큰은 출력하지 않는다.
SSH 배너 모의는 `SSH_LAB_HOST`에 실습 서버를 지정했을 때만 실행한다.

In [ ]:
%pip install -q paramiko

In [3]:
import base64
import getpass
import json
import re
import secrets
import shlex
import time
from contextlib import contextmanager
from datetime import datetime, timezone

import paramiko

DEV_HOST = input("dev-server Tailscale IP/host: ").strip()
DEV_USER = input("dev-server user: ").strip()
DEV_PASS = getpass.getpass("dev-server password: ")
def normalize_access_token(raw):
    value = raw.strip()
    if value.startswith("{"):
        try:
            value = str(json.loads(value).get("accessToken", "")).strip()
        except json.JSONDecodeError:
            pass
    if len(value) >= 2 and value[0] == value[-1] and value[0] in "\"'":
        value = value[1:-1].strip()
    if value.lower().startswith("bearer "):
        value = value[7:].strip()
    return value


def read_access_token(prompt):
    for attempt in range(3):
        value = normalize_access_token(getpass.getpass(prompt))
        parts = value.split(".")
        if len(parts) == 3 and all(parts):
            return value
        if attempt < 2:
            print("JWT access token 형식이 아닙니다. 로그인 응답의 accessToken을 입력하세요.")
            prompt = "새 access token: "
    raise RuntimeError("JWT access token을 입력하지 않았습니다.")


ACCESS_TOKEN = read_access_token("테스트 계정 access token: ")

NS = "deepmesh"
VAGRANT_DIR = "~/k8s-cluster"
WIN = 5
SSH_LAB_HOST = ""  # 직접 관리하는 실습 서버
PIN = {}  # 예: {"post": "post-service-..."}
MAIN_CTR = {"auth": "auth-service", "post": "post-service",
            "comment": "comment-service", "frontend": "frontend"}
PORT = {"auth": 8080, "post": 8080, "comment": 8080, "frontend": 80}
EW_HEADERS = {"User-Agent": "Apache-HttpClient/5.3.1 (Java/17.0.9)",
              "Accept": "application/json"}
BR_HEADERS = {"User-Agent": "Mozilla/5.0 DeepMeshDemo/1.0",
              "Accept": "application/json, text/html;q=0.9, */*;q=0.8"}

In [4]:
def dev_run(command, timeout=180):
    client = paramiko.SSHClient()
    client.load_system_host_keys()
    client.set_missing_host_key_policy(paramiko.AutoAddPolicy())
    try:
        client.connect(DEV_HOST, username=DEV_USER, password=DEV_PASS, timeout=15)
        stdin, stdout, stderr = client.exec_command(command, timeout=timeout)
        stdin.close()
        channel = stdout.channel
        out, err = [], []
        deadline = time.monotonic() + timeout
        while True:
            while channel.recv_ready():
                out.append(channel.recv(65536))
            while channel.recv_stderr_ready():
                err.append(channel.recv_stderr(65536))
            if channel.exit_status_ready() and not channel.recv_ready() and not channel.recv_stderr_ready():
                break
            if time.monotonic() > deadline:
                raise TimeoutError("SSH 명령 시간 초과")
            time.sleep(0.02)
        return {"rc": channel.recv_exit_status(),
                "out": b"".join(out).decode(errors="replace"),
                "err": b"".join(err).decode(errors="replace")}
    finally:
        client.close()


def node_run(vm, script, timeout=180):
    encoded = base64.b64encode(script.encode()).decode()
    inner = f"printf %s {shlex.quote(encoded)} | base64 -d | bash"
    command = f"cd {VAGRANT_DIR} && vagrant ssh {shlex.quote(vm)} -c {shlex.quote(inner)}"
    result = dev_run(command, timeout)
    if result["rc"]:
        # 원격 명령에는 토큰이 들어갈 수 있어 명령문 전체는 출력하지 않는다.
        raise RuntimeError(f"{vm}: 원격 명령 실패(rc={result['rc']})")
    return result["out"]


def pod_shell(svc, script, pod=None, timeout=180):
    selected = pod or POD[svc]
    encoded = base64.b64encode(script.encode()).decode()
    inner = f"printf %s {shlex.quote(encoded)} | base64 -d | sh"
    command = (f"kubectl -n {shlex.quote(NS)} exec {shlex.quote(selected['name'])} "
               f"-c {shlex.quote(MAIN_CTR[svc])} -- sh -c {shlex.quote(inner)}")
    return node_run("k8s-master", command, timeout)


def decode_pod_output(output):
    start, end = "__DM_PODS_B64__", "__DM_END__"
    for section in reversed(output.split(start)[1:]):
        if end not in section:
            continue
        encoded = "".join(section.split(end, 1)[0].split())
        try:
            text = base64.b64decode(encoded, validate=True).decode()
        except (ValueError, UnicodeDecodeError):
            continue
        items = []
        for line in text.splitlines():
            fields = line.split("\t")
            if len(fields) != 5:
                items = []
                break
            name, ip, node, ready, containers = fields
            items.append({
                "metadata": {"name": name},
                "spec": {"nodeName": node, "containers": [
                    {"name": value} for value in containers.split(",") if value
                ]},
                "status": {"podIP": ip, "conditions": [
                    {"type": "Ready", "status": ready}
                ]},
            })
        if items:
            return {"items": items}
    raise RuntimeError(
        f"pod 목록 응답을 해석하지 못했습니다 "
        f"(시작 표식 {output.count(start)}개, 종료 표식 {output.count(end)}개)."
    )


def discover_and_pin():
    template = (r'{{range .items}}{{.metadata.name}}{{"\t"}}{{.status.podIP}}{{"\t"}}'
                r'{{.spec.nodeName}}{{"\t"}}{{range .status.conditions}}'
                r'{{if eq .type "Ready"}}{{.status}}{{end}}{{end}}{{"\t"}}'
                r'{{range .spec.containers}}{{.name}},{{end}}{{"\n"}}{{end}}')
    command = ("tmp=$(mktemp); trap 'rm -f \"$tmp\"' EXIT; "
               f"kubectl -n {shlex.quote(NS)} get pods "
               f"-o {shlex.quote('go-template=' + template)} > \"$tmp\"; rc=$?; "
               "printf '__DM_PODS_B64__'; "
               "if [ \"$rc\" -eq 0 ]; then base64 < \"$tmp\" | tr -d '\\r\\n'; fi; "
               "printf '\\n__DM_END__\\n'; exit \"$rc\"")
    output = node_run("k8s-master", command)
    data = decode_pod_output(output)
    pod, sibling = {}, {}
    for svc, container in MAIN_CTR.items():
        candidates = []
        for item in data["items"]:
            status = item.get("status", {})
            ready = any(c.get("type") == "Ready" and c.get("status") == "True"
                        for c in status.get("conditions", []))
            names = {c["name"] for c in item["spec"]["containers"]}
            if container in names and ready and status.get("podIP"):
                candidates.append({"name": item["metadata"]["name"], "ip": status["podIP"],
                                   "node": item["spec"]["nodeName"]})
        candidates.sort(key=lambda p: p["name"])
        chosen = next((p for p in candidates if p["name"] == PIN.get(svc)), None)
        if PIN.get(svc) and chosen is None:
            raise RuntimeError(f"{svc}: 지정한 pod가 Ready 상태가 아님")
        if not candidates:
            raise RuntimeError(f"{svc}: Ready pod 없음")
        pod[svc] = chosen or candidates[0]
        others = [p for p in candidates if p["name"] != pod[svc]["name"]]
        if others:
            sibling[svc] = others[0]
        print(f"{svc:9s} {pod[svc]['name']}  sibling={others[0]['name'] if others else '-'}")
    return pod, sibling


def sibling_of(svc):
    if svc not in SIB:
        raise RuntimeError(f"{svc}: 이 시나리오에는 형제 replica가 필요함")
    return SIB[svc]


def url(svc, path="", pod=None):
    selected = pod or POD[svc]
    return f"http://{selected['ip']}:{PORT[svc]}{path}"


POD, SIB = discover_and_pin()

auth      auth-service-54f4ff846c-hkzwt  sibling=auth-service-54f4ff846c-pwnkw
post      post-service-dc8cbdfc7-2zqsg  sibling=post-service-dc8cbdfc7-h8xtr
comment   comment-service-69bd896c9c-6q78x  sibling=comment-service-69bd896c9c-trqs6
frontend  frontend-f94f6666c-7brz6  sibling=frontend-f94f6666c-fmsrz


## 2. 요청 실행

요청별 HTTP 상태·연결 수·source port와 전체 curl 종료 코드를 출력한다.
응답 본문은 메모리에 보관하며, 오류 코드와 메시지만 화면에 표시한다.
`--next`마다 HTTP/TLS/인증 옵션을 다시 지정한다.

In [5]:
API_SETUP = r'''
TOKEN_PATH=/var/run/secrets/kubernetes.io/serviceaccount/token
CA=/var/run/secrets/kubernetes.io/serviceaccount/ca.crt
test -s "$TOKEN_PATH" && test -s "$CA" || { echo "service account token/CA 없음" >&2; exit 2; }
TOKEN=$(cat "$TOKEN_PATH")
AS="https://${KUBERNETES_SERVICE_HOST:-kubernetes.default.svc}:${KUBERNETES_SERVICE_PORT_HTTPS:-443}"
'''


def request(target, method="GET", headers=None, data=None, api=False):
    return {"url": target, "method": method, "headers": dict(headers or {}),
            "data": data, "api": api}


def make_http_body(requests):
    if not requests:
        raise ValueError("요청이 비어 있음")
    lines = [
        'set -eu', 'umask 077', 'command -v curl >/dev/null',
        'D=$(mktemp -d)',
        'trap \'rm -rf -- "$D"\' EXIT',
    ]
    if any(r["api"] for r in requests):
        lines.append(API_SETUP)
    lines.append("set --")
    for i, item in enumerate(requests):
        if item["data"] is not None:
            raw = json.dumps(item["data"], ensure_ascii=False).encode()
            encoded = base64.b64encode(raw).decode()
            lines.append(f'printf %s {shlex.quote(encoded)} | base64 -d > "$D/input{i}"')
        if i:
            lines.append('set -- "$@" --next')
        # 각 전송에 옵션을 반복하고 source port는 OS에 맡긴다.
        options = ["--silent", "--show-error", "--http1.1", "--noproxy", "*",
                   "--connect-timeout", "4", "--max-time", "12", "--request", item["method"]]
        headers = {"Accept": "application/json", **item["headers"]}
        for key, value in headers.items():
            if any(c in key + value for c in "\r\n"):
                raise ValueError("HTTP 헤더에 줄바꿈을 넣을 수 없음")
            options += ["--header", f"{key}: {value}"]
        if item["data"] is not None:
            options += ["--header", "Content-Type: application/json"]
        lines.append('set -- "$@" ' + " ".join(shlex.quote(v) for v in options))
        if item["api"]:
            if not item["url"].startswith("/") or item["url"].startswith("//"):
                raise ValueError("API 요청은 절대 경로를 사용")
            lines.append('set -- "$@" --cacert "$CA" --header "Authorization: Bearer $TOKEN"')
        if item["data"] is not None:
            lines.append(f'set -- "$@" --data-binary "@$D/input{i}"')
        fmt = (f"__DM_HTTP__|{i}|%{{http_code}}|%{{local_port}}|%{{remote_ip}}|"
               "%{remote_port}|%{num_connects}|%{time_total}|%{size_download}\\n")
        target = '"$AS"' + shlex.quote(item["url"]) if item["api"] else shlex.quote(item["url"])
        lines.append(f'set -- "$@" --output "$D/body{i}" --write-out {shlex.quote(fmt)} {target}')
    lines += ['set +e', 'curl "$@" 2>"$D/curl.err"', 'rc=$?', 'set -e',
              'printf "__DM_CURL_RC__|%s\\n" "$rc"']
    for i in range(len(requests)):
        lines += [f'printf "__DM_BODY__|{i}|"',
                  f'if test -f "$D/body{i}"; then base64 < "$D/body{i}" | tr -d "\\r\\n"; fi',
                  'printf "\\n"']
    lines += ['printf "__DM_STDERR__|"', 'base64 < "$D/curl.err" | tr -d "\\r\\n"',
              'printf "\\n"']
    return "\n".join(lines) + "\n"


def parse_http_output(output, count):
    rows = {i: {"index": i, "http": 0, "local_port": 0, "new_connections": 0,
                "body": "", "seen": False} for i in range(count)}
    curl_rc, stderr = None, ""
    for line in output.splitlines():
        if line.startswith("__DM_HTTP__|"):
            _, idx, http, local, remote, port, opened, elapsed, size = line.split("|")
            row = rows[int(idx)]
            row.update(http=int(http), local_port=int(local or 0), remote=remote,
                       remote_port=int(port or 0), new_connections=int(opened or 0),
                       seconds=float(elapsed), size=int(float(size)), seen=True)
        elif line.startswith("__DM_BODY__|"):
            _, idx, body = line.split("|", 2)
            rows[int(idx)]["body"] = base64.b64decode(body).decode(errors="replace")
        elif line.startswith("__DM_CURL_RC__|"):
            curl_rc = int(line.split("|")[1])
        elif line.startswith("__DM_STDERR__|"):
            stderr = base64.b64decode(line.split("|", 1)[1]).decode(errors="replace")
    if curl_rc is None:
        raise RuntimeError("curl 결과 표식을 받지 못함")
    return {"rows": list(rows.values()), "curl_rc": curl_rc, "stderr": stderr}


RESULTS = []


def run_http(title, exec_svc, watch_svc, requests, external=False, quiet=False):
    body = make_http_body(requests)
    started = datetime.now(timezone.utc).isoformat(timespec="seconds")
    if not quiet:
        print(f"\n[{title}] 시작 {started} / 관측={watch_svc}")
    timeout = len(requests) * 12 + 120
    if external:
        output = node_run(POD[exec_svc]["node"], body, timeout)
    else:
        output = pod_shell(exec_svc, body, timeout=timeout)
    result = parse_http_output(output, len(requests))
    result.update(title=title, started=started, ended=datetime.now(timezone.utc).isoformat(timespec="seconds"),
                  watch=watch_svc)
    if not quiet:
        for row in result["rows"]:
            label = requests[row["index"]]["url"]
            print(f"  {row['index'] + 1:2d} HTTP={row['http']:03d} "
                  f"src-port={row['local_port']} new-conn={row['new_connections']} {label}")
            try:
                error = json.loads(row["body"])
            except (ValueError, TypeError):
                error = {}
            if isinstance(error, dict) and row["http"] >= 400:
                code = error.get("errorCode") or error.get("error") or ""
                message = error.get("message") or error.get("reason") or ""
                print("     ", str(code), str(message)[:160])
        missing = sum(not r["seen"] or r["http"] == 0 for r in result["rows"])
        print(f"  종료 {result['ended']} / curl rc={result['curl_rc']} / HTTP 응답 없음={missing}")
        if result["stderr"]:
            print("  curl:", result["stderr"].strip()[:400])
        print("  탐지 여부와 윈도우 수는 이 시간대의 대시보드에서 확인")
    RESULTS.append(result)
    return result


def json_response(result, expected):
    if result["curl_rc"] or len(result["rows"]) != 1:
        raise RuntimeError("단일 요청 전송 실패")
    row = result["rows"][0]
    if row["http"] not in expected:
        raise RuntimeError(f"HTTP {row['http']} (기대: {sorted(expected)})")
    return json.loads(row["body"]) if row["body"] else {}


def auth_headers():
    if not ACCESS_TOKEN:
        raise RuntimeError("테스트 계정의 access token을 먼저 입력")
    return {**EW_HEADERS, "Authorization": f"Bearer {ACCESS_TOKEN}"}


def api_call(svc, path, method="GET", data=None, expected=(200,)):
    result = run_http("테스트 데이터 준비/정리", svc, svc,
                      [request(url(svc, path), method, auth_headers(), data)],
                      external=True, quiet=True)
    return json_response(result, set(expected))


def validate_access_token():
    global ACCESS_TOKEN
    for attempt in range(3):
        result = run_http("access token 확인", "auth", "auth",
                          [request(url("auth", "/internal/auth/validate"),
                                   headers=auth_headers())],
                          external=True, quiet=True)
        row = result["rows"][0]
        if not result["curl_rc"] and row["seen"] and row["http"] == 200:
            return json_response(result, {200})
        if not result["curl_rc"] and row["seen"] and row["http"] == 401:
            if attempt == 2:
                break
            print("access token이 거부되었습니다. 로그인 응답의 accessToken을 다시 입력하세요.")
            ACCESS_TOKEN = read_access_token("새 access token: ")
            continue
        return json_response(result, {200})
    raise RuntimeError("access token 검증에 3회 실패했습니다. 로그인 후 새 토큰으로 다시 실행하세요.")


USER = validate_access_token()
if not isinstance(USER.get("userId"), int) or not USER.get("username"):
    raise RuntimeError("토큰 검증 응답 형식 확인 필요")
print("토큰 검증 완료: userId =", USER["userId"])

토큰 검증 완료: userId = 17273


## 3. 테스트 데이터

조회·삭제 셀은 이 노트북에서 만든 글과 댓글을 사용한다.
`l3`는 실제 DELETE다. 기존 ID를 넣지 않고, 셀 종료 시 테스트 글을 정리한다.
정리 실패 시 ID를 남기므로 마지막 정리 셀을 다시 실행한다.

In [4]:
RUN_TAG = globals().get("RUN_TAG", "deepmesh-demo-" + secrets.token_hex(4))
TEST_POSTS = globals().get("TEST_POSTS", {})
R1_BACKUPS = globals().get("R1_BACKUPS", [])


def is_deepmesh_block(row):
    if row["http"] != 403:
        return False
    try:
        body = json.loads(row["body"])
    except (TypeError, ValueError):
        return False
    return body.get("error") == "blocked" and "deepmesh" in body.get("reason", "").lower()


def ensure_status(result, expected, allow_block=True):
    missing = [r["index"] + 1 for r in result["rows"] if not r["seen"]]
    blocked = [r for r in result["rows"] if is_deepmesh_block(r)]
    bad = [r["index"] + 1 for r in result["rows"]
           if r["seen"] and r["http"] not in expected
           and not (allow_block and is_deepmesh_block(r))]
    if result["curl_rc"] or missing or bad:
        raise RuntimeError(
            f"요청 결과 확인 필요: curl={result['curl_rc']}, "
            f"응답 없음={missing}, 예상 밖 응답={bad}"
        )
    if blocked:
        print("DeepMesh 차단:", [r["index"] + 1 for r in blocked])
    return {"expected": [r for r in result["rows"] if r["http"] in expected],
            "blocked": blocked}


def cleanup_posts(ids=None):
    failed = []
    for post_id in list(TEST_POSTS if ids is None else ids):
        if post_id not in TEST_POSTS:
            continue
        try:
            post = api_call("post", f"/api/posts/{post_id}", expected=(200, 404))
            if post.get("errorCode") == "POST_NOT_FOUND":
                TEST_POSTS.pop(post_id)
                continue
            if post.get("title") != TEST_POSTS[post_id] or post.get("userId") != USER["userId"]:
                raise RuntimeError("테스트 글의 소유자/표식 불일치")
            api_call("post", f"/api/posts/{post_id}", "DELETE")
            TEST_POSTS.pop(post_id)
            print("테스트 글 정리:", post_id)
        except Exception as exc:
            failed.append(post_id)
            print(f"정리 실패: postId={post_id} ({type(exc).__name__})")
    if failed:
        raise RuntimeError(f"남은 테스트 글: {failed}")


@contextmanager
def fixture_posts(count=WIN):
    ids = []
    try:
        for _ in range(count):
            title = RUN_TAG + "-" + secrets.token_hex(3)
            post = api_call("post", "/api/posts", "POST",
                            {"title": title, "content": "DeepMesh 시연용 글"}, expected=(201,))
            post_id = post["postId"]
            TEST_POSTS[post_id] = title
            ids.append(post_id)
            api_call("comment", f"/api/comments/{post_id}/comments", "POST",
                     {"content": "DeepMesh 시연용 댓글"}, expected=(201,))
        print("테스트 글 준비:", ids)
        yield sorted(ids)
    finally:
        cleanup_posts(ids)


def confirm_deleted_comments(ids):
    for post_id in ids:
        body = api_call("comment", f"/api/comments/{post_id}/comments?size=10")
        if body.get("data") != []:
            raise RuntimeError(f"댓글 삭제 미확인: postId={post_id}")
    if ids:
        print("테스트 댓글 삭제 확인:", ids)


def confirm_retained_comments(ids):
    for post_id in ids:
        body = api_call("comment", f"/api/comments/{post_id}/comments?size=10")
        if not body.get("data"):
            raise RuntimeError(f"차단된 DELETE의 댓글이 남아 있지 않음: postId={post_id}")
    if ids:
        print("차단된 DELETE의 댓글 보존 확인:", ids)

## 4. 정상 트래픽

post·comment의 validate는 실제 토큰으로 응답을 확인한다. 403 차단은 정상 요청 오탐으로 기록한다.
auth 익명 refresh는 쿠키가 없으므로 401이 정상이며, frontend는 SPA HTML 응답을 확인한다.

In [5]:
with fixture_posts(1) as ids:
    post_id = ids[0]
    listing = api_call("comment", f"/api/comments/{post_id}/comments?size=10")
    comments = [item for item in listing.get("data", [])
                if item.get("postId") == post_id and item.get("userId") == USER["userId"]]
    if len(comments) != 1:
        raise RuntimeError("시연용 댓글을 확인하지 못함")
    comment_id = comments[0]["commentId"]
    cases = [
        ("post", f"/api/posts/{post_id}",
         {"title": TEST_POSTS[post_id], "content": "DeepMesh 시연용 글"}, "postId", post_id),
        ("comment", f"/api/comments/{comment_id}",
         {"content": "DeepMesh 시연용 댓글"}, "commentId", comment_id),
    ]
    for svc, path, payload, id_key, item_id in cases:
        BODY = [request(url(svc, path), "PUT",
                        {**BR_HEADERS, "Authorization": f"Bearer {ACCESS_TOKEN}"}, payload)
                for _ in range(WIN)]
        result = run_http(f"{svc} benign — 수정 API의 정상 토큰 검증",
                          svc, svc, BODY, external=True)
        ensure_status(result, {200}, allow_block=False)
        for row in result["rows"]:
            body = json.loads(row["body"])
            if body.get("userId") != USER["userId"] or body.get(id_key) != item_id:
                raise RuntimeError("테스트 데이터의 사용자 또는 ID 불일치")
            if any(body.get(key) != value for key, value in payload.items()):
                raise RuntimeError("테스트 데이터 수정 결과 불일치")

테스트 글 준비: [7697]

[post benign — 수정 API의 정상 토큰 검증] 시작 2026-09-18T05:11:32+00:00 / 관측=post
   1 HTTP=200 src-port=33206 new-conn=1 http://10.244.194.123:8080/api/posts/7697
   2 HTTP=200 src-port=33206 new-conn=0 http://10.244.194.123:8080/api/posts/7697
   3 HTTP=200 src-port=33206 new-conn=0 http://10.244.194.123:8080/api/posts/7697
   4 HTTP=200 src-port=33206 new-conn=0 http://10.244.194.123:8080/api/posts/7697
   5 HTTP=200 src-port=33206 new-conn=0 http://10.244.194.123:8080/api/posts/7697
  종료 2026-09-18T05:11:41+00:00 / curl rc=0 / HTTP 응답 없음=0
  탐지 여부와 윈도우 수는 이 시간대의 대시보드에서 확인

[comment benign — 수정 API의 정상 토큰 검증] 시작 2026-09-18T05:11:41+00:00 / 관측=comment
   1 HTTP=200 src-port=40912 new-conn=1 http://10.244.126.17:8080/api/comments/11903
   2 HTTP=200 src-port=40912 new-conn=0 http://10.244.126.17:8080/api/comments/11903
   3 HTTP=200 src-port=40912 new-conn=0 http://10.244.126.17:8080/api/comments/11903
   4 HTTP=200 src-port=40912 new-conn=0 http://10.244.126.17:8080/api/comme

In [6]:
with fixture_posts(1) as ids:
    post_id = ids[0]
    BODY = [request(url("comment", f"/api/comments/{post_id}/comments?size=10"),
                    headers=BR_HEADERS) for _ in range(WIN)]
    result = run_http("comment benign — 댓글 조회의 글 존재 확인",
                      "comment", "comment", BODY, external=True)
    ensure_status(result, {200}, allow_block=False)
    for row in result["rows"]:
        body = json.loads(row["body"])
        if body.get("postId") != post_id or not body.get("data"):
            raise RuntimeError("시연용 게시글 또는 댓글 조회 실패")

테스트 글 준비: [7699]

[comment benign — 댓글 조회의 글 존재 확인] 시작 2026-09-18T05:12:38+00:00 / 관측=comment
   1 HTTP=200 src-port=52920 new-conn=1 http://10.244.126.17:8080/api/comments/7699/comments?size=10
   2 HTTP=200 src-port=52920 new-conn=0 http://10.244.126.17:8080/api/comments/7699/comments?size=10
   3 HTTP=200 src-port=52920 new-conn=0 http://10.244.126.17:8080/api/comments/7699/comments?size=10
   4 HTTP=200 src-port=52920 new-conn=0 http://10.244.126.17:8080/api/comments/7699/comments?size=10
   5 HTTP=200 src-port=52920 new-conn=0 http://10.244.126.17:8080/api/comments/7699/comments?size=10
  종료 2026-09-18T05:12:50+00:00 / curl rc=0 / HTTP 응답 없음=0
  탐지 여부와 윈도우 수는 이 시간대의 대시보드에서 확인
테스트 글 정리: 7699


In [ ]:
BODY = [request(url("auth", "/api/auth/refresh"), "POST", BR_HEADERS) for _ in range(WIN)]
result = run_http("auth benign — 쿠키 없는 refresh 응답", "auth", "auth", BODY, external=True)
ensure_status(result, {401}, allow_block=False)

In [8]:
routes = ["/", "/posts", "/posts/17", "/posts/new", "/auth/sign-in"]
BODY = [request(url("frontend", path + "?demo=" + secrets.token_hex(4)), headers=BR_HEADERS)
        for path in routes]
result = run_http("frontend benign — SPA 응답", "frontend", "frontend", BODY, external=True)
ensure_status(result, {200}, allow_block=False)
if not all("<html" in r["body"].lower() for r in result["rows"]):
    raise RuntimeError("SPA HTML 응답이 아님")


[frontend benign — SPA 응답] 시작 2026-09-18T05:13:23+00:00 / 관측=frontend
   1 HTTP=200 src-port=48064 new-conn=1 http://10.244.100.249:80/?demo=116345dc
   2 HTTP=200 src-port=48064 new-conn=0 http://10.244.100.249:80/posts?demo=e25433ce
   3 HTTP=200 src-port=48064 new-conn=0 http://10.244.100.249:80/posts/17?demo=e0893fa3
   4 HTTP=200 src-port=48064 new-conn=0 http://10.244.100.249:80/posts/new?demo=2cbd2d68
   5 HTTP=200 src-port=48064 new-conn=0 http://10.244.100.249:80/auth/sign-in?demo=fe835fea
  종료 2026-09-18T05:13:31+00:00 / curl rc=0 / HTTP 응답 없음=0
  탐지 여부와 윈도우 수는 이 시간대의 대시보드에서 확인


## 5. l2 — 잘못된 토큰의 인증 시도

실제 토큰의 서명을 바꾸거나 잘못된 JWT를 보낸다. 유효 계정 탈취나 토큰 재사용 성공을 뜻하지 않는다.
현재 컨버터는 JWT 서명을 판별하지 않으므로, 서버의 401과 DeepMesh의 403 차단을 구분한다.

In [9]:
def invalid_tokens():
    parts = ACCESS_TOKEN.split(".")
    if len(parts) != 3 or not parts[2]:
        raise RuntimeError("서명된 JWT 형식이 아님")
    # 서명의 첫 문자를 바꿔 마지막 base64 패딩 비트만 달라지는 경우를 피한다.
    signature = ("A" if parts[2][0] != "A" else "B") + parts[2][1:]
    changed = ".".join([parts[0], parts[1], signature])
    header = base64.urlsafe_b64encode(b'{"alg":"none"}').decode().rstrip("=")
    unsigned = f"{header}.{parts[1]}."
    return [changed, unsigned, "invalid-demo-token", changed, unsigned]


for svc in ("post", "comment"):
    BODY = [request(url("auth", "/internal/auth/validate"),
                    headers={**EW_HEADERS, "Authorization": f"Bearer {token}"})
            for token in invalid_tokens()]
    result = run_http(f"{svc} l2 — 잘못된 토큰", svc, svc, BODY)
    ensure_status(result, {401})


[post l2 — 잘못된 토큰] 시작 2026-09-18T05:13:38+00:00 / 관측=post
   1 HTTP=401 src-port=47890 new-conn=1 http://10.244.100.248:8080/internal/auth/validate
      TOKEN_INVALID 유효하지 않은 토큰입니다.
   2 HTTP=401 src-port=47890 new-conn=0 http://10.244.100.248:8080/internal/auth/validate
      TOKEN_INVALID 유효하지 않은 토큰입니다.
   3 HTTP=401 src-port=47890 new-conn=0 http://10.244.100.248:8080/internal/auth/validate
      TOKEN_INVALID 유효하지 않은 토큰입니다.
   4 HTTP=403 src-port=47890 new-conn=0 http://10.244.100.248:8080/internal/auth/validate
      blocked anomalous request dropped by deepmesh
   5 HTTP=403 src-port=47890 new-conn=0 http://10.244.100.248:8080/internal/auth/validate
      blocked anomalous request dropped by deepmesh
  종료 2026-09-18T05:13:47+00:00 / curl rc=0 / HTTP 응답 없음=0
  탐지 여부와 윈도우 수는 이 시간대의 대시보드에서 확인
DeepMesh 차단: [4, 5]

[comment l2 — 잘못된 토큰] 시작 2026-09-18T05:13:47+00:00 / 관측=comment
   1 HTTP=401 src-port=34496 new-conn=1 http://10.244.100.248:8080/internal/auth/validate
      TOKEN_INVA

## 6. enum_seq — 댓글 순차 조회

기존 `validate?userId=` 대신 실제 댓글 조회 API를 사용한다.
테스트 글 ID를 증가하는 순서로 조회한다. 공개 데이터 조회이므로 요청만으로 공격을 확정할 수 없다.

In [10]:
with fixture_posts() as ids:
    BODY = [request(url("comment", f"/api/comments/{post_id}/comments?size=10"),
                    headers=BR_HEADERS) for post_id in ids]
    result = run_http("post enum_seq — 댓글 순차 조회 모의", "post", "post", BODY)
    ensure_status(result, {200})

테스트 글 준비: [7701, 7702, 7703, 7704, 7706]

[post enum_seq — 댓글 순차 조회 모의] 시작 2026-09-18T05:15:11+00:00 / 관측=post
   1 HTTP=200 src-port=60598 new-conn=1 http://10.244.126.17:8080/api/comments/7701/comments?size=10
   2 HTTP=200 src-port=60598 new-conn=0 http://10.244.126.17:8080/api/comments/7702/comments?size=10
   3 HTTP=200 src-port=60598 new-conn=0 http://10.244.126.17:8080/api/comments/7703/comments?size=10
   4 HTTP=403 src-port=60598 new-conn=0 http://10.244.126.17:8080/api/comments/7704/comments?size=10
      blocked anomalous request dropped by deepmesh
   5 HTTP=403 src-port=60598 new-conn=0 http://10.244.126.17:8080/api/comments/7706/comments?size=10
      blocked anomalous request dropped by deepmesh
  종료 2026-09-18T05:15:23+00:00 / curl rc=0 / HTTP 응답 없음=0
  탐지 여부와 윈도우 수는 이 시간대의 대시보드에서 확인
DeepMesh 차단: [4, 5]
테스트 글 정리: 7701
테스트 글 정리: 7702
테스트 글 정리: 7703
테스트 글 정리: 7704
테스트 글 정리: 7706


In [11]:
peer = sibling_of("comment")
with fixture_posts() as ids:
    BODY = [request(url("comment", f"/api/comments/{post_id}/comments?size=10", pod=peer),
                    headers=BR_HEADERS) for post_id in ids]
    result = run_http("comment enum_seq — 형제 replica 순차 조회", "comment", "comment", BODY)
    ensure_status(result, {200})

테스트 글 준비: [7708, 7709, 7710, 7711, 7712]

[comment enum_seq — 형제 replica 순차 조회] 시작 2026-09-18T05:17:49+00:00 / 관측=comment
   1 HTTP=200 src-port=46922 new-conn=1 http://10.244.100.251:8080/api/comments/7708/comments?size=10
   2 HTTP=200 src-port=46922 new-conn=0 http://10.244.100.251:8080/api/comments/7709/comments?size=10
   3 HTTP=403 src-port=46922 new-conn=0 http://10.244.100.251:8080/api/comments/7710/comments?size=10
      blocked anomalous request dropped by deepmesh
   4 HTTP=403 src-port=46922 new-conn=0 http://10.244.100.251:8080/api/comments/7711/comments?size=10
      blocked anomalous request dropped by deepmesh
   5 HTTP=403 src-port=46922 new-conn=0 http://10.244.100.251:8080/api/comments/7712/comments?size=10
      blocked anomalous request dropped by deepmesh
  종료 2026-09-18T05:18:01+00:00 / curl rc=0 / HTTP 응답 없음=0
  탐지 여부와 윈도우 수는 이 시간대의 대시보드에서 확인
DeepMesh 차단: [3, 4, 5]
테스트 글 정리: 7708
테스트 글 정리: 7709
테스트 글 정리: 7710
테스트 글 정리: 7711
테스트 글 정리: 7712


## 7. l3 — 내부 DELETE 남용 모의

토큰 없이 내부 삭제 API를 호출해 테스트 댓글을 실제로 삭제한다.
정상 게시글 삭제에도 쓰이는 API다. 여기서는 게시글 삭제 없이 댓글만 일괄 삭제하는 상황을 만든다.
204면 삭제 여부를, DeepMesh 403이면 댓글이 남아 있는지를 확인한다.

In [12]:
with fixture_posts() as ids:
    BODY = [request(url("comment", f"/internal/posts/{post_id}/comments"),
                    "DELETE", EW_HEADERS) for post_id in ids]
    result = run_http("post l3 — 테스트 댓글 일괄 삭제", "post", "post", BODY)
    status = ensure_status(result, {204})
    deleted = [post_id for post_id, row in zip(ids, result["rows"]) if row["http"] == 204]
    blocked = [post_id for post_id, row in zip(ids, result["rows"]) if is_deepmesh_block(row)]
    confirm_deleted_comments(deleted)
    confirm_retained_comments(blocked)

테스트 글 준비: [7714, 7716, 7717, 7718, 7719]

[post l3 — 테스트 댓글 일괄 삭제] 시작 2026-09-18T05:20:26+00:00 / 관측=post
   1 HTTP=204 src-port=56556 new-conn=1 http://10.244.126.17:8080/internal/posts/7714/comments
   2 HTTP=204 src-port=56556 new-conn=0 http://10.244.126.17:8080/internal/posts/7716/comments
   3 HTTP=204 src-port=56556 new-conn=0 http://10.244.126.17:8080/internal/posts/7717/comments
   4 HTTP=204 src-port=56556 new-conn=0 http://10.244.126.17:8080/internal/posts/7718/comments
   5 HTTP=204 src-port=56556 new-conn=0 http://10.244.126.17:8080/internal/posts/7719/comments
  종료 2026-09-18T05:20:38+00:00 / curl rc=0 / HTTP 응답 없음=0
  탐지 여부와 윈도우 수는 이 시간대의 대시보드에서 확인
테스트 댓글 삭제 확인: [7714, 7716, 7717, 7718, 7719]
테스트 글 정리: 7714
테스트 글 정리: 7716
테스트 글 정리: 7717
테스트 글 정리: 7718
테스트 글 정리: 7719


In [14]:
peer = sibling_of("comment")
with fixture_posts() as ids:
    BODY = [request(url("comment", f"/internal/posts/{post_id}/comments", pod=peer),
                    "DELETE", EW_HEADERS) for post_id in ids]
    result = run_http("comment l3 — 형제 replica 내부 DELETE", "comment", "comment", BODY)
    status = ensure_status(result, {204})
    deleted = [post_id for post_id, row in zip(ids, result["rows"]) if row["http"] == 204]
    blocked = [post_id for post_id, row in zip(ids, result["rows"]) if is_deepmesh_block(row)]
    confirm_deleted_comments(deleted)
    confirm_retained_comments(blocked)

테스트 글 준비: [7735, 7736, 7737, 7738, 7739]

[comment l3 — 형제 replica 내부 DELETE] 시작 2026-09-18T05:38:12+00:00 / 관측=comment
   1 HTTP=204 src-port=36248 new-conn=1 http://10.244.100.251:8080/internal/posts/7735/comments
   2 HTTP=204 src-port=36248 new-conn=0 http://10.244.100.251:8080/internal/posts/7736/comments
   3 HTTP=204 src-port=36248 new-conn=0 http://10.244.100.251:8080/internal/posts/7737/comments
   4 HTTP=403 src-port=36248 new-conn=0 http://10.244.100.251:8080/internal/posts/7738/comments
      blocked anomalous request dropped by deepmesh
   5 HTTP=403 src-port=36248 new-conn=0 http://10.244.100.251:8080/internal/posts/7739/comments
      blocked anomalous request dropped by deepmesh
  종료 2026-09-18T05:38:23+00:00 / curl rc=0 / HTTP 응답 없음=0
  탐지 여부와 윈도우 수는 이 시간대의 대시보드에서 확인
DeepMesh 차단: [4, 5]
테스트 댓글 삭제 확인: [7735, 7736, 7737]
차단된 DELETE의 댓글 보존 확인: [7738, 7739]
테스트 글 정리: 7735
테스트 글 정리: 7736
테스트 글 정리: 7737
테스트 글 정리: 7738
테스트 글 정리: 7739


## 8. cred_enum — 비밀번호 추측과 계정 응답 비교

테스트 계정과 존재하지 않을 것으로 예상되는 이름에 각각 5번 요청한다.
실패 메시지 차이는 계정 열거에 쓰일 수 있다. auth 모델은 자격증명 내용 대신 flow 특징을 본다.

In [15]:
peer = sibling_of("auth")
missing_user = "dm_missing_" + secrets.token_hex(3)
names = [USER["username"], missing_user] * 5
BODY = [request(url("auth", "/api/auth/login", pod=peer), "POST", BR_HEADERS,
                {"username": name, "password": "wrong-" + secrets.token_hex(16)})
        for name in names]
result = run_http("auth cred_enum — 제한된 비밀번호 추측", "auth", "auth", BODY)
ensure_status(result, {401})
for name, row in zip(names, result["rows"]):
    body = json.loads(row["body"])
    message = "DeepMesh 차단" if is_deepmesh_block(row) else body.get("message")
    print(name, ":", message)


[auth cred_enum — 제한된 비밀번호 추측] 시작 2026-09-18T05:45:12+00:00 / 관측=auth
   1 HTTP=403 src-port=52582 new-conn=1 http://10.244.126.11:8080/api/auth/login
      blocked anomalous request dropped by deepmesh
   2 HTTP=403 src-port=52582 new-conn=0 http://10.244.126.11:8080/api/auth/login
      blocked anomalous request dropped by deepmesh
   3 HTTP=403 src-port=52582 new-conn=0 http://10.244.126.11:8080/api/auth/login
      blocked anomalous request dropped by deepmesh
   4 HTTP=403 src-port=52582 new-conn=0 http://10.244.126.11:8080/api/auth/login
      blocked anomalous request dropped by deepmesh
   5 HTTP=403 src-port=52582 new-conn=0 http://10.244.126.11:8080/api/auth/login
      blocked anomalous request dropped by deepmesh
   6 HTTP=403 src-port=52582 new-conn=0 http://10.244.126.11:8080/api/auth/login
      blocked anomalous request dropped by deepmesh
   7 HTTP=403 src-port=52582 new-conn=0 http://10.244.126.11:8080/api/auth/login
      blocked anomalous request dropped by deepmes

## 9. scan_seq와 정상·스캔 혼합

`/.env`, `/.git/config` 등의 민감 경로를 탐색한다. 404나 403도 탐색 시도의 응답이다.
혼합 셀은 정상 글 조회 4개 사이에 스캔 요청 1개를 넣는다.

In [16]:
paths = ["/.env", "/.git/config", "/actuator/env", "/admin", "/swagger-ui/index.html"]
BODY = [request(url("post", path), headers=BR_HEADERS) for path in paths]
result = run_http("frontend scan_seq — backend 경로 탐색", "frontend", "frontend", BODY)


[frontend scan_seq — backend 경로 탐색] 시작 2026-09-18T05:45:26+00:00 / 관측=frontend
   1 HTTP=404 src-port=60646 new-conn=1 http://10.244.194.123:8080/.env
      ENDPOINT_NOT_FOUND 요청한 경로를 찾을 수 없습니다.
   2 HTTP=404 src-port=60646 new-conn=0 http://10.244.194.123:8080/.git/config
      ENDPOINT_NOT_FOUND 요청한 경로를 찾을 수 없습니다.
   3 HTTP=403 src-port=60646 new-conn=0 http://10.244.194.123:8080/actuator/env
      blocked anomalous request dropped by deepmesh
   4 HTTP=403 src-port=60646 new-conn=0 http://10.244.194.123:8080/admin
      blocked anomalous request dropped by deepmesh
   5 HTTP=403 src-port=60646 new-conn=0 http://10.244.194.123:8080/swagger-ui/index.html
      blocked anomalous request dropped by deepmesh
  종료 2026-09-18T05:45:34+00:00 / curl rc=0 / HTTP 응답 없음=0
  탐지 여부와 윈도우 수는 이 시간대의 대시보드에서 확인


## 10. r1 — HTML 변조와 스크립트 삽입 응답

현재 `index.html`을 별도 파일로 백업하고 실행 표식용 스크립트를 앞에 붙인다.
curl 응답에서 삽입을 확인한 뒤 `finally`에서 원본을 복원한다.
이 셀은 쿠키 탈취나 브라우저 JavaScript 실행을 검증하지 않는다.

In [18]:
R1_MARKER = "deepmesh-demo-script"
R1_SCRIPT = '<script>/*deepmesh-demo-script*/window.__deepmesh_demo_xss__=true;</script>\n'


def make_r1_prepare(record):
    src = "/usr/share/nginx/html/index.html"
    backup, prefix = record["backup"], record["prefix"]
    encoded = base64.b64encode(R1_SCRIPT.encode()).decode()
    return "\n".join([
        "set -eu",
        f"test ! -e {shlex.quote(backup)}",
        f"cp -p {shlex.quote(src)} {shlex.quote(backup)}",
        f"printf %s {shlex.quote(encoded)} | base64 -d > {shlex.quote(prefix)}",
        f"cat {shlex.quote(prefix)} {shlex.quote(backup)} > {shlex.quote(src)}",
        f"grep -Fq {shlex.quote(R1_MARKER)} {shlex.quote(src)}",
    ])


def restore_frontend(record):
    src = "/usr/share/nginx/html/index.html"
    backup, prefix = map(shlex.quote, (record["backup"], record["prefix"]))
    body = (
        f"set -eu\nif test -f {backup}; then\n"
        f"  cp -p {backup} {shlex.quote(src)}\n"
        f"  cmp -s {backup} {shlex.quote(src)}\n"
        f"  rm -f {backup} {prefix}\n"
        "  echo '__DM_RESTORE__|restored'\n"
        "else\n  echo 'HTML 백업 없음' >&2\n  exit 2\nfi\n"
    )
    output = pod_shell("frontend", body, pod=record["pod"])
    if "__DM_RESTORE__|restored" not in output:
        raise RuntimeError("HTML 복원 결과 미확인")
    R1_BACKUPS.remove(record)
    print("HTML 복원:", output.split("__DM_RESTORE__|", 1)[1].splitlines()[0])


def run_r1():
    if any(r["pod"]["name"] == POD["frontend"]["name"] for r in R1_BACKUPS):
        raise RuntimeError("남은 HTML 백업을 마지막 정리 셀에서 먼저 복원")
    stem = "/tmp/deepmesh-r1-" + secrets.token_hex(6)
    record = {"pod": dict(POD["frontend"]), "backup": stem + ".bak", "prefix": stem + ".prefix"}
    R1_BACKUPS.append(record)
    print("HTML 백업:", record["pod"]["name"], record["backup"])
    try:
        pod_shell("frontend", make_r1_prepare(record), pod=record["pod"])
        BODY = [request(url("frontend", "/"), headers={**BR_HEADERS, "Accept": "text/html"})
                for _ in range(WIN)]
        result = run_http("frontend r1 — 스크립트 삽입 응답", "post", "frontend", BODY)
        ensure_status(result, {200})
        delivered = sum(R1_MARKER in r["body"] for r in result["rows"])
        replaced = len(result["rows"]) - delivered
        print(f"변조 응답 전달={delivered}, 깨끗한 응답으로 대체={replaced}")
        print("브라우저 JavaScript 실행 여부는 측정하지 않음.")
    finally:
        restore_frontend(record)


run_r1()

HTML 백업: frontend-f94f6666c-7brz6 /tmp/deepmesh-r1-c2d6b11c1137.bak

[frontend r1 — 스크립트 삽입 응답] 시작 2026-09-18T05:46:28+00:00 / 관측=frontend
   1 HTTP=200 src-port=60078 new-conn=1 http://10.244.100.249:80/
   2 HTTP=200 src-port=60078 new-conn=0 http://10.244.100.249:80/
   3 HTTP=200 src-port=60078 new-conn=0 http://10.244.100.249:80/
   4 HTTP=200 src-port=49132 new-conn=1 http://10.244.100.249:80/
   5 HTTP=200 src-port=49144 new-conn=1 http://10.244.100.249:80/
  종료 2026-09-18T05:46:40+00:00 / curl rc=0 / HTTP 응답 없음=0
  탐지 여부와 윈도우 수는 이 시간대의 대시보드에서 확인
변조 응답 전달=3, 깨끗한 응답으로 대체=2
브라우저 JavaScript 실행 여부는 측정하지 않음.
HTML 복원: restored


## 11. Kubernetes API

pod의 ServiceAccount 토큰과 CA 인증서를 사용한다. TLS 오류는 HTTP 000, 권한 거부는 보통 403이다.
`k1`은 정찰 모의이며, 승인된 운영 도구의 같은 조회까지 공격으로 단정하지 않는다.
`k2`는 post pod에서 실행하는 dry-run으로 실제 배포·삭제·컨테이너 명령 실행은 일어나지 않는다.

In [8]:
def run_k8s_flow(title, svc, requests, payload=None):
    body = r'''
TOKEN=$(cat /var/run/secrets/kubernetes.io/serviceaccount/token 2>/dev/null)
AS="https://${KUBERNETES_SERVICE_HOST:-kubernetes.default.svc}:${KUBERNETES_SERVICE_PORT_HTTPS:-443}"
codes=""
rc_all=0

'''

    for index, path in enumerate(requests):
        method = "GET"
        data = ""
        if payload is not None and index == 0:
            method = "POST"
            data = f''' -H "Content-Type: application/json" --data '{json.dumps(payload, separators=(",", ":"))}' '''

        body += f'''
code=$(curl -sS -k --http1.1 --connect-timeout 3 --max-time 5 \
  -H "Authorization: Bearer $TOKEN" \
  -X {method}{data} \
  -o /dev/null -w "%{{http_code}}" \
  "$AS{path}" 2>/dev/null)
rc=$?
codes="$codes $code"
if [ "$rc" -ne 0 ]; then rc_all=$rc; fi
'''

    body += r'''
printf '[k8s] HTTP codes:%s (curl exit %s)\n' "$codes" "$rc_all"
exit 0
'''

    print(f"\n[{title}]")
    print(pod_shell(svc, body))

In [19]:
paths = [
    "/api/v1/namespaces/" + NS + "/pods",
    "/api/v1/namespaces/" + NS + "/secrets",
    "/version",
    "/api/v1/namespaces/" + NS + "/pods",
    "/api/v1/namespaces/" + NS + "/secrets",
    "/version",
    "/api/v1/namespaces/" + NS + "/pods",
    "/api/v1/namespaces/" + NS + "/secrets",
]

run_k8s_flow(
    "post k1 — Kubernetes API 정찰",
    "post",
    paths,
)


[post k1 — Kubernetes API 정찰]
[k8s] HTTP codes: 403 403 000 403 000 200 000 403 (curl exit 35)



In [18]:
name = "deepmesh-dryrun-" + secrets.token_hex(4)
base = f"/api/v1/namespaces/{NS}"

pod_spec = {
    "apiVersion": "v1",
    "kind": "Pod",
    "metadata": {"name": name},
    "spec": {
        "containers": [
            {
                "name": "demo",
                "image": "busybox:1.36",
                "command": ["sleep", "60"],
            }
        ]
    },
}

paths = [
    base + "/pods?dryRun=All",
    base + "/configmaps?dryRun=All",
    base + f"/pods/{name}?dryRun=All",
    "/version",
    base + "/pods",
    base + "/secrets",
    base + "/serviceaccounts",
    base + "/configmaps",
]

run_k8s_flow(
    "comment k2 — Kubernetes 리소스 조작",
    "comment",
    paths,
    payload=pod_spec,
)


[comment k2 — Kubernetes 리소스 조작]
[k8s] HTTP codes: 403 403 403 200 403 403 000 403 (curl exit 35)



## 12. SSH 연결·배너 모의 (선택)

`SSH_LAB_HOST:22`에 한 번 연결해 tmate 형태의 SSH 배너를 보낸다.
도구 설치, SSH 인증, 원격 제어는 하지 않는다. 기존의 공용 IP 목록은 사용하지 않는다.

In [ ]:
def make_ssh_body(host):
    if not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9.-]*", host):
        raise ValueError("실습용 IPv4 주소 또는 DNS 이름을 입력하세요")

    return f'''
set -u
HOST={shlex.quote(host)}
count=0

for i in $(seq 1 8); do
    (
        timeout 4 bash -c '{
            'exec 3<>/dev/tcp/"$1"/22 && '
            'printf "SSH-2.0-tmate_demo\\\\r\\\\n" >&3 && '
            'head -c 200 <&3 >/dev/null'
        }' _ "$HOST"
    ) >/dev/null 2>&1 &
    count=$((count + 1))
done

wait 2>/dev/null
printf "[ssh] 연결·배너 시도 %s회 완료\\n" "$count"
'''

SSH_TARGET = ""

print("SSH 연결 시도 시작:", datetime.now(timezone.utc).isoformat(timespec="seconds"))
print(pod_shell("auth", make_ssh_body(SSH_TARGET)))

SSH 연결 시도 시작: 2026-09-18T09:38:50+00:00
[ssh] 연결·배너 시도 8회 완료



## 13. 남은 테스트 데이터 정리

일반 예외에서는 각 셀의 정리 코드가 실행된다. 커널 종료나 통신 장애 후에는 원격 상태를 별도로 확인한다.
글 삭제가 실패하면 `TEST_POSTS`, HTML 복원이 실패하면 `R1_BACKUPS`에 대상이 남는다.
DB·로그에 아무 변화도 없었다는 의미의 net-zero는 주장하지 않는다.

In [23]:
cleanup_errors = []
for record in list(R1_BACKUPS):
    try:
        restore_frontend(record)
    except Exception as exc:
        cleanup_errors.append(f"HTML: {record['pod']['name']} / {record['backup']}")
        print("HTML 복원 실패:", type(exc).__name__)
try:
    cleanup_posts()
except Exception:
    cleanup_errors.append(f"테스트 글: {list(TEST_POSTS)}")

if cleanup_errors:
    raise RuntimeError("정리 재시도 필요: " + "; ".join(cleanup_errors))
print("추적 중인 테스트 글과 HTML 백업 정리 완료")

테스트 글 정리: 7721
추적 중인 테스트 글과 HTML 백업 정리 완료


## 결과 확인

- `error=blocked`인 403은 DeepMesh가 요청을 차단한 결과다.
- 정상 시나리오의 차단은 오탐, 공격 시나리오의 차단은 방어 성공으로 해석한다.
- 애플리케이션의 401/403/404와 전송 성공, 공격 성공을 구분한다.
- 000은 HTTP 응답을 받지 못한 경우다. curl 오류와 pod 상태를 확인한다.
- l2는 서버 인증 실패, l3는 테스트 댓글 삭제, r1은 응답 변조를 각각 확인한다.
- 탐지 결과는 시나리오 시간대의 서비스·방향·목적지를 기준으로 대시보드와 대조한다.

참고: [curl의 --next 옵션](https://curl.se/docs/manpage.html#-:),
[Kubernetes dry-run](https://kubernetes.io/docs/reference/using-api/api-concepts/#dry-run).